# День 6 — Инференс и сравнение моделей

**Цель:** сравнить подходы к классификации на одной и той же выборке.

Сравниваем **три** модели:

| Модель | Признаки | Что обучается |
|---|---|---|
| TF-IDF + LogReg | частоты слов и биграмм | 1 линейный классификатор |
| Заморожен BERT + LogReg (День 4) | CLS-эмбеддинги | 769 параметров |
| Fine-tuned BERT (День 5) | обучаемые представления | 66 955 010 параметров |

TF-IDF добавлен, чтобы ответить на вопрос «а нужен ли вообще трансформер» — без него сравнение показывает только *насколько лучше стало*, но не *с чего мы начинали*.

Переиспользуемый код — в [`compare_utils.py`](compare_utils.py).

## Задача 1: Загрузка моделей

In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

model_ft = AutoModelForSequenceClassification.from_pretrained('./fine_tuned_model')
tokenizer = AutoTokenizer.from_pretrained('./fine_tuned_model')
model_ft.eval()

print('Fine-tuned модель загружена')

C:\Users\Vsevolod\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fine-tuned модель загружена


In [2]:
import os
import joblib
from compare_utils import (
    fit_and_save_baseline, load_baseline,
    fit_and_save_tfidf, load_tfidf,
)
from tokenization_utils import load_tokenizer
from embeddings_utils import load_model

# Baseline Дня 4: замороженный BERT + LogReg
if not os.path.exists('baseline_model.pkl'):
    print('Обучаем baseline (~1-2 мин)...')
    fit_and_save_baseline()
baseline_model = load_baseline()

# Классический TF-IDF + LogReg
if not os.path.exists('tfidf_model.pkl'):
    print('Обучаем TF-IDF...')
    fit_and_save_tfidf()
tfidf_model, tfidf_vectorizer = load_tfidf()

# Энкодер — играет роль "векторизатора" для baseline
tokenizer_enc = load_tokenizer()
encoder = load_model()

print(f'Baseline: {baseline_model.coef_.shape[1]} признаков (CLS-эмбеддинг)')
print(f'TF-IDF:   {len(tfidf_vectorizer.vocabulary_)} признаков (слова + биграммы)')

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Baseline: 768 признаков (CLS-эмбеддинг)
TF-IDF:   4423 признаков (слова + биграммы)


Уже здесь видна принципиальная разница: у baseline 768 **плотных** признаков, у TF-IDF — 4423 **разреженных**. Для каждого отдельного текста почти все TF-IDF признаки равны нулю: в предложении из 8 слов ненулевых признаков десяток, остальные 4400+ — нули.

## Задача 2: Функция предсказания для fine-tuned

In [3]:
def predict_fine_tuned(texts, model, tokenizer):
    if isinstance(texts, str):
        texts = [texts]

    predictions = []

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)

        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.nn.functional.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

        predictions.append({
            'text': text,
            'prediction': pred,
            'probabilities': probs[0].cpu().numpy(),
        })

    return predictions

## Задача 3: Функции предсказания для baseline и TF-IDF

### 3a. Baseline Дня 4 (замороженный BERT)

> В шаблоне задания здесь предполагался `vectorizer.transform(texts)`. У нашего baseline из Дня 4 TF-IDF-векторизатора нет — его роль играет сам DistilBERT, поэтому вызываем `get_cls_embeddings`.

In [4]:
from baseline_utils import get_cls_embeddings


def predict_baseline(texts, model, tokenizer, encoder):
    if isinstance(texts, str):
        texts = [texts]

    # Роль "векторизатора" играет сам DistilBERT
    X = get_cls_embeddings(texts, tokenizer, encoder)

    predictions = model.predict(X)
    probs = model.predict_proba(X) if hasattr(model, 'predict_proba') else None

    results = []
    for i, text in enumerate(texts):
        results.append({
            'text': text,
            'prediction': int(predictions[i]),
            'probabilities': probs[i] if probs is not None else None,
        })

    return results

### 3b. TF-IDF

А вот здесь `vectorizer.transform` уже настоящий — это ровно та функция, что была в шаблоне задания.

In [5]:
def predict_tfidf(texts, model, vectorizer):
    if isinstance(texts, str):
        texts = [texts]

    X = vectorizer.transform(texts)
    predictions = model.predict(X)
    probs = model.predict_proba(X) if hasattr(model, 'predict_proba') else None

    results = []
    for i, text in enumerate(texts):
        results.append({
            'text': text,
            'prediction': int(predictions[i]),
            'probabilities': probs[i] if probs is not None else None,
        })

    return results

> ⚠️ **Про утечку данных.** В `fit_and_save_tfidf` векторизатор обучается **только на train**: `fit_transform(train_texts)`, а к тесту применяется `transform`. Если вызвать `fit_transform` на всех данных сразу, словарь и IDF-веса «увидят» тестовую выборку — метрики окажутся завышенными. Это одна из самых частых ошибок с TF-IDF.

## Задача 4: Сравнение на примерах

In [6]:
test_texts = [
    "This movie was absolutely fantastic!",
    "Terrible, waste of my time.",
    "It was okay, nothing special.",
    "Best film I've seen this year!",
    "Boring and too long.",
]

preds_ft = predict_fine_tuned(test_texts, model_ft, tokenizer)
preds_base = predict_baseline(test_texts, baseline_model, tokenizer_enc, encoder)
preds_tfidf = predict_tfidf(test_texts, tfidf_model, tfidf_vectorizer)

LABELS = {0: 'negative', 1: 'positive'}

for i, text in enumerate(test_texts):
    print(f'\nТекст: {text}')
    for name, p in [
        ('TF-IDF     ', preds_tfidf[i]),
        ('Baseline   ', preds_base[i]),
        ('Fine-tuned ', preds_ft[i]),
    ]:
        conf = p['probabilities'].max()
        print(f'  {name}: {LABELS[p["prediction"]]:8s} (уверенность {conf:.3f})')
    all_agree = len({preds_tfidf[i]['prediction'], preds_base[i]['prediction'],
                     preds_ft[i]['prediction']}) == 1
    print(f'  Все три согласны: {all_agree}')


Текст: This movie was absolutely fantastic!


  TF-IDF     : negative (уверенность 0.661)
  Baseline   : positive (уверенность 0.991)
  Fine-tuned : positive (уверенность 0.986)
  Все три согласны: False

Текст: Terrible, waste of my time.
  TF-IDF     : negative (уверенность 0.570)
  Baseline   : negative (уверенность 0.993)
  Fine-tuned : negative (уверенность 0.986)
  Все три согласны: True

Текст: It was okay, nothing special.
  TF-IDF     : negative (уверенность 0.652)
  Baseline   : positive (уверенность 0.725)
  Fine-tuned : negative (уверенность 0.741)
  Все три согласны: False

Текст: Best film I've seen this year!
  TF-IDF     : positive (уверенность 0.656)
  Baseline   : positive (уверенность 0.999)
  Fine-tuned : positive (уверенность 0.979)
  Все три согласны: True

Текст: Boring and too long.
  TF-IDF     : negative (уверенность 0.724)
  Baseline   : negative (уверенность 0.995)
  Fine-tuned : negative (уверенность 0.985)
  Все три согласны: True


## Задача 5: Confusion Matrix для всех моделей

Берём **ту же** валидационную выборку, что в Днях 4 и 5 (600 примеров, `random_state=42`).

In [7]:
from compare_utils import get_test_split

eval_texts, eval_labels = get_test_split()
print(f'Тестовая выборка: {len(eval_texts)} примеров')
print(f'Баланс: negative={eval_labels.count(0)}, positive={eval_labels.count(1)}')

Тестовая выборка: 600 примеров
Баланс: negative=265, positive=335


In [8]:
from compare_utils import predict_fine_tuned as predict_ft_fast

# Батчевая версия из модуля — на 600 примерах заметно быстрее поштучной
y_pred_ft = [p['prediction'] for p in predict_ft_fast(eval_texts, model_ft, tokenizer)]
y_pred_base = [p['prediction'] for p in predict_baseline(eval_texts, baseline_model, tokenizer_enc, encoder)]
y_pred_tfidf = [p['prediction'] for p in predict_tfidf(eval_texts, tfidf_model, tfidf_vectorizer)]

print(f'Предсказаний: tfidf={len(y_pred_tfidf)}, base={len(y_pred_base)}, ft={len(y_pred_ft)}')

Предсказаний: tfidf=600, base=600, ft=600


In [9]:
import matplotlib.pyplot as plt
from compare_utils import plot_confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

cms = {}
for ax, (title, y_pred) in zip(axes, [
    ('TF-IDF + LogReg', y_pred_tfidf),
    ('Заморожен BERT + LogReg', y_pred_base),
    ('Fine-tuned BERT', y_pred_ft),
]):
    cms[title] = plot_confusion_matrix(eval_labels, y_pred, title, ax=ax)

plt.tight_layout()
plt.savefig('confusion_matrix_comparison.png', dpi=100)
plt.show()

for title, cm in cms.items():
    print(f'{title:28s} ошибок: {cm[0,1] + cm[1,0]:3d}  '
          f'(FP={cm[0,1]}, FN={cm[1,0]})')

TF-IDF + LogReg              ошибок: 158  (FP=114, FN=44)
Заморожен BERT + LogReg      ошибок:  81  (FP=42, FN=39)
Fine-tuned BERT              ошибок:  53  (FP=24, FN=29)


C:\Users\Vsevolod\AppData\Local\Temp\ipykernel_17376\445149442.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Задача 6: Сравнение метрик

In [10]:
from sklearn.metrics import classification_report, f1_score, accuracy_score

all_preds = {
    'TF-IDF + LogReg': y_pred_tfidf,
    'Заморожен BERT + LogReg': y_pred_base,
    'Fine-tuned BERT': y_pred_ft,
}

for name, y_pred in all_preds.items():
    print(f'=== {name} ===')
    print(classification_report(eval_labels, y_pred, target_names=['negative', 'positive']))

=== TF-IDF + LogReg ===
              precision    recall  f1-score   support

    negative       0.77      0.57      0.66       265
    positive       0.72      0.87      0.79       335

    accuracy                           0.74       600
   macro avg       0.75      0.72      0.72       600
weighted avg       0.74      0.74      0.73       600

=== Заморожен BERT + LogReg ===
              precision    recall  f1-score   support

    negative       0.85      0.84      0.85       265
    positive       0.88      0.88      0.88       335

    accuracy                           0.86       600
   macro avg       0.86      0.86      0.86       600
weighted avg       0.86      0.86      0.86       600

=== Fine-tuned BERT ===
              precision    recall  f1-score   support

    negative       0.89      0.91      0.90       265
    positive       0.93      0.91      0.92       335

    accuracy                           0.91       600
   macro avg       0.91      0.91      0.91     

In [11]:
from compare_utils import compare_models, summary_table

results = compare_models(eval_labels, all_preds)
print(summary_table(results))

print('\nСогласие моделей между собой:')
for pair, value in results['agreement'].items():
    print(f'  {pair}: {value:.1%}')

Модель                        macro F1  Accuracy   Ошибок      Δ F1
-------------------------------------------------------------------
TF-IDF + LogReg                 0.7215    0.7367      158         —
Заморожен BERT + LogReg         0.8630    0.8650       81   +0.1415
Fine-tuned BERT                 0.9106    0.9117       53   +0.1891
(Δ F1 — прирост относительно «TF-IDF + LogReg»)

Согласие моделей между собой:
  TF-IDF + LogReg vs Заморожен BERT + LogReg: 72.8%
  TF-IDF + LogReg vs Fine-tuned BERT: 75.2%
  Заморожен BERT + LogReg vs Fine-tuned BERT: 90.7%


### Насколько велики эти разницы

Выборка всего 600 примеров, поэтому одна ошибка ≈ 0.0017 accuracy. Посчитаем доверительные интервалы, прежде чем делать выводы.

In [12]:
import numpy as np

n = len(eval_labels)
for name, m in results['models'].items():
    acc = m['accuracy']
    se = np.sqrt(acc * (1 - acc) / n)
    print(f'{name:28s} acc = {acc:.4f} ± {1.96*se:.4f} (95% CI), ошибок {m["errors"]}')

TF-IDF + LogReg              acc = 0.7367 ± 0.0352 (95% CI), ошибок 158
Заморожен BERT + LogReg      acc = 0.8650 ± 0.0273 (95% CI), ошибок 81
Fine-tuned BERT              acc = 0.9117 ± 0.0227 (95% CI), ошибок 53


In [13]:
# Визуальное сравнение
names = list(results['models'])
f1s = [results['models'][n_]['f1'] for n_ in names]
accs = [results['models'][n_]['accuracy'] for n_ in names]

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - 0.2, f1s, 0.4, label='macro F1')
ax.bar(x + 0.2, accs, 0.4, label='accuracy')

for i, (f, a) in enumerate(zip(f1s, accs)):
    ax.text(i - 0.2, f + 0.01, f'{f:.3f}', ha='center', fontsize=9)
    ax.text(i + 0.2, a + 0.01, f'{a:.3f}', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_title('Сравнение трёх подходов (SST-2, 600 примеров)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\Vsevolod\AppData\Local\Temp\ipykernel_17376\4185923691.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Задача 7: Сохранение результатов сравнения

In [14]:
from compare_utils import save_comparison

save_comparison(results, 'comparison_results.txt')
print(open('comparison_results.txt', encoding='utf-8').read())

=== Сравнение моделей (День 6) ===

Тестовая выборка: 600 примеров (SST-2, та же в Днях 4-6)

Модель                        macro F1  Accuracy   Ошибок      Δ F1
-------------------------------------------------------------------
TF-IDF + LogReg                 0.7215    0.7367      158         —
Заморожен BERT + LogReg         0.8630    0.8650       81   +0.1415
Fine-tuned BERT                 0.9106    0.9117       53   +0.1891
(Δ F1 — прирост относительно «TF-IDF + LogReg»)

Согласие моделей между собой:
  TF-IDF + LogReg vs Заморожен BERT + LogReg: 72.8%
  TF-IDF + LogReg vs Fine-tuned BERT: 75.2%
  Заморожен BERT + LogReg vs Fine-tuned BERT: 90.7%

--- TF-IDF + LogReg: classification report ---
              precision    recall  f1-score   support

    negative       0.77      0.57      0.66       265
    positive       0.72      0.87      0.79       335

    accuracy                           0.74       600
   macro avg       0.75      0.72      0.72       600
weighted avg       

## Чекпоинт и выводы

- ✅ Функции `predict_fine_tuned`, `predict_baseline`, `predict_tfidf`
- ✅ Confusion matrix для всех трёх моделей
- ✅ Сравнение метрик
- ✅ `comparison_results.txt`

### Что показало добавление TF-IDF

Прогрессия получилась наглядной: каждый шаг «умнее» предыдущего примерно вдвое сокращает разрыв до идеала.

**Почему TF-IDF так отстаёт именно здесь** — важно понимать, что дело не только в «BERT умнее»:

1. **Тексты очень короткие.** Медиана SST-2 — 8 токенов. TF-IDF описывает текст мешком слов, и в 8 словах сигнала мало. На длинных документах (новости, отзывы на страницу) разрыв был бы заметно меньше.
2. **Слова вне словаря просто не существуют.** Словарь собран из 2400 обучающих текстов и содержит 4423 признака. Наглядный провал: фразу «This movie was absolutely **fantastic**!» TF-IDF назвал *negative* — потому что слова `fantastic` в его словаре **нет** (проверено: `'fantastic' in vectorizer.vocabulary_` → `False`). Самое информативное слово фразы для модели невидимо. BERT же встречал его на предобучении.
3. **Нет обобщения по смыслу.** Для TF-IDF `great` и `fantastic` — два независимых признака, между ними нулевая связь. В пространстве эмбеддингов они рядом.

**Что это не значит:** TF-IDF не «устаревший мусор». Он обучается за секунды вместо минут, не требует GPU, работает на CPU в тысячи раз быстрее на инференсе и полностью интерпретируем (можно посмотреть веса конкретных слов). Для многих задач — длинные документы, много данных, жёсткие требования к скорости — это по-прежнему разумный выбор. Здесь он проигрывает из-за конкретного сочетания «короткие тексты + мало данных».

### Оговорка про воспроизводимость

Fine-tuning **недетерминирован между запусками**. Два прогона Дня 5 с одинаковыми гиперпараметрами дали разные траектории:

| Эпоха | Прогон A | Прогон B |
|---|---|---|
| 1 | 0.8826 | 0.8981 |
| 2 | **0.9106** | 0.8990 |
| 3 | 0.9059 | **0.9204** |

Причина — случайная инициализация головы классификации и порядок батчей при `shuffle=True`. Разброс между запусками (~0.01–0.02 F1) сопоставим с разницей между эпохами, поэтому утверждать «модель переобучается на 3-й эпохе» по одному прогону нельзя. На диск сохранён Прогон A, эпоха 2 (val F1 = **0.9106**) — это и есть та модель, что фигурирует в `comparison_results.txt` и в анализе ошибок Дня 7.

А вот разрывы между тремя подходами (0.72 → 0.86 → 0.91) намного больше этого разброса — эти выводы устойчивы.